# REU Assignment 1

**Author:** Akbar Aman  
**Date:** 6/19/26

## Problem statement

This assignment tasked us to investigate how two hyperparameters, learning rate and network size, affected the performance of a convolutional neural network on Fashion-MNIST. For every configuration, I report training and validation running time, loss, and accuracy. I then compared the configurations, identified the observed trends, and explained the results in terms of convergence speed, optimization stability, model capacity, underfitting, and overfitting.

## Material provided

I was provided with a starter notebook containing Fashion-MNIST loading and normalization, a fixed CNN with two convolutional layers and two fully connected layers, cross-entropy loss, SGD with momentum, a 10-epoch training loop, test accuracy calculation, and sample prediction visualization. The starter code used one learning rate and one network size. Therefore, I extended it to perform controlled comparisons, create a validation split, report complete timing and metric tables, and analyze the experimental trends.

## Response to the assignment requirements

1. I created one fixed 50,000/10,000 train-validation split and reserved the original 10,000-image test set for one final evaluation.
2. I tested learning rates `0.0001`, `0.001`, and `0.01` while holding the baseline network size fixed.
3. I tested small `(16, 32, 64)`, baseline `(32, 64, 128)`, and large `(64, 128, 256)` networks while holding the learning rate at `0.001`.
4. I reported cumulative training time, cumulative validation time, final training loss and accuracy, final validation loss and accuracy, parameter count, and the train-validation accuracy gap for every configuration.
5. I plotted validation loss and accuracy across epochs to examine convergence speed and stability.
6. I derived the comparison statements from measured results, selected the best model using validation accuracy, and evaluated that model once on the untouched test set.

## Objective

I designed this controlled experiment to isolate the effects of learning rate and network size on CNN performance. I trained each configuration for 10 epochs using SGD with momentum.

## Experimental controls

- I used the same random 50,000/10,000 train-validation split in every run.
- I fixed the random seeds, batch size, optimizer, momentum, epoch count, preprocessing, and data order.
- I defined the baseline architecture as `(32, 64, 128)`, denoting the two convolution widths and hidden-layer width.
- I used `0.001` as the baseline learning rate.
- I excluded the test set from tuning and evaluated it once after validation-based model selection.
- Timing was hardware-dependent, so time comparisons were valid only within this execution.

## Execution environment

I executed the experiment in Google Colab using a T4 GPU and the PyTorch runtime provided by Colab. Fashion-MNIST was downloaded programmatically by the data-preparation cell. I executed all cells sequentially in one runtime session so that every configuration was measured under the same hardware and software conditions.


In [ ]:
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

SEED = 1024
BATCH_SIZE = 64
EPOCHS = 10
BASELINE_SIZE = (32, 64, 128)
BASELINE_LR = 0.001

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

split_generator = torch.Generator().manual_seed(SEED)
train_set, val_set = torch.utils.data.random_split(
    full_train, [50_000, 10_000], generator=split_generator
)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
}
val_loader = torch.utils.data.DataLoader(val_set, shuffle=False, **loader_options)
test_loader = torch.utils.data.DataLoader(test_set, shuffle=False, **loader_options)
train_eval_loader = torch.utils.data.DataLoader(train_set, shuffle=False, **loader_options)

def make_train_loader():
    generator = torch.Generator().manual_seed(SEED)
    return torch.utils.data.DataLoader(
        train_set, shuffle=True, generator=generator, **loader_options
    )

print(f"Train: {len(train_set):,}, validation: {len(val_set):,}, test: {len(test_set):,}")


In [ ]:
class ConfigurableCNN(nn.Module):
    def __init__(self, channels_1, channels_2, hidden_units):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, channels_1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channels_1, channels_2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels_2 * 7 * 7, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

criterion = nn.CrossEntropyLoss()


In [ ]:
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

def train_epoch(model, loader, optimizer):
    model.train()
    loss_sum = 0.0
    correct = 0
    sample_count = 0
    synchronize()
    start = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += outputs.argmax(dim=1).eq(labels).sum().item()
        sample_count += batch_size

    synchronize()
    elapsed = time.perf_counter() - start
    return loss_sum / sample_count, correct / sample_count, elapsed

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    correct = 0
    sample_count = 0
    synchronize()
    start = time.perf_counter()

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        batch_size = labels.size(0)
        loss_sum += loss.item() * batch_size
        correct += outputs.argmax(dim=1).eq(labels).sum().item()
        sample_count += batch_size

    synchronize()
    elapsed = time.perf_counter() - start
    return loss_sum / sample_count, correct / sample_count, elapsed


In [ ]:
def run_experiment(name, learning_rate, network_size):
    seed_everything()
    model = ConfigurableCNN(*network_size).to(device)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    train_loader = make_train_loader()
    history = []

    for epoch in range(1, EPOCHS + 1):
        online_loss, online_accuracy, train_time = train_epoch(model, train_loader, optimizer)
        val_loss, val_accuracy, val_time = evaluate(model, val_loader)
        history.append({
            "epoch": epoch,
            "train_loss": online_loss,
            "train_accuracy": online_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "train_time_s": train_time,
            "val_time_s": val_time,
        })
        print(
            f"{name:18s} | epoch {epoch:2d}/{EPOCHS} | "
            f"train loss {online_loss:.4f}, acc {online_accuracy:.2%} | "
            f"val loss {val_loss:.4f}, acc {val_accuracy:.2%}"
        )

    train_loss, train_accuracy, train_eval_time = evaluate(model, train_eval_loader)
    history_frame = pd.DataFrame(history)
    final_epoch = history_frame.iloc[-1]
    summary = {
        "configuration": name,
        "learning_rate": learning_rate,
        "network_size": str(network_size),
        "parameters": parameter_count(model),
        "train_time_s": history_frame["train_time_s"].sum(),
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_time_s": history_frame["val_time_s"].sum(),
        "validation_loss": final_epoch["val_loss"],
        "validation_accuracy": final_epoch["val_accuracy"],
        "generalization_gap_pp": 100 * (train_accuracy - final_epoch["val_accuracy"]),
        "train_evaluation_time_s": train_eval_time,
    }
    return summary, history_frame, copy.deepcopy(model.state_dict())


## Hyperparameter configurations

For the learning-rate study, I held the architecture fixed. For the network-size study, I held the learning rate fixed. I executed the shared baseline once, producing five unique runs rather than six.


In [ ]:
experiments = [
    ("LR 0.0001", 0.0001, BASELINE_SIZE),
    ("Baseline", BASELINE_LR, BASELINE_SIZE),
    ("LR 0.01", 0.01, BASELINE_SIZE),
    ("Small network", BASELINE_LR, (16, 32, 64)),
    ("Large network", BASELINE_LR, (64, 128, 256)),
]

summaries = []
histories = {}
states = {}

for name, learning_rate, network_size in experiments:
    summary, history, state = run_experiment(name, learning_rate, network_size)
    summaries.append(summary)
    histories[name] = history
    states[name] = state

results = pd.DataFrame(summaries)
display_columns = [
    "configuration", "learning_rate", "network_size", "parameters",
    "train_time_s", "train_loss", "train_accuracy",
    "validation_time_s", "validation_loss", "validation_accuracy",
    "generalization_gap_pp",
]
results[display_columns].style.format({
    "learning_rate": "{:.4g}",
    "parameters": "{:,}",
    "train_time_s": "{:.2f}",
    "train_loss": "{:.4f}",
    "train_accuracy": "{:.2%}",
    "validation_time_s": "{:.3f}",
    "validation_loss": "{:.4f}",
    "validation_accuracy": "{:.2%}",
    "generalization_gap_pp": "{:.2f}",
}).hide(axis="index")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, history in histories.items():
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", label=name)
    axes[1].plot(history["epoch"], 100 * history["val_accuracy"], marker="o", label=name)

axes[0].set(title="Validation loss by epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Validation accuracy by epoch", xlabel="Epoch", ylabel="Accuracy (%)")
for axis in axes:
    axis.grid(alpha=0.3)
axes[1].legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Results and interpretation

I derived each claim directly from the measured results. Accuracy alone was insufficient for a rigorous comparison. Validation loss reflected the quality and confidence of predictions, the train-validation gap provided evidence of overfitting, and parameter count together with runtime quantified computational cost.

| Configuration | Learning rate | Network size | Parameters | Train time (s) | Train loss | Train accuracy | Validation time (s) | Validation loss | Validation accuracy | Gap (pp) |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|
| LR 0.0001 | 0.0001 | (32, 64, 128) | 421,642 | 123.02 | 0.5239 | 81.35% | 20.956 | 0.5356 | 80.65% | 0.70 |
| Baseline | 0.001 | (32, 64, 128) | 421,642 | 114.97 | 0.2969 | 89.49% | 18.209 | 0.3196 | 88.77% | 0.72 |
| LR 0.01 | 0.01 | (32, 64, 128) | 421,642 | 115.31 | 0.1206 | 95.60% | 17.993 | 0.2314 | 91.96% | 3.64 |
| Small network | 0.001 | (16, 32, 64) | 105,866 | 115.24 | 0.3153 | 88.69% | 19.906 | 0.3381 | 88.35% | 0.34 |
| Large network | 0.001 | (64, 128, 256) | 1,682,954 | 116.98 | 0.2817 | 90.00% | 18.454 | 0.3084 | 88.99% | 1.01 |

### Learning-rate comparison

The best tested learning rate was `0.01`, which achieved 91.96% validation accuracy and 0.2314 validation loss. Increasing the learning rate from `0.0001` to `0.01` improved final validation accuracy by 11.31 percentage points within the fixed 10-epoch budget. The `0.0001` model was still improving at epoch 10, indicating that it converged too slowly for the allotted training period. The `0.01` model converged fastest and produced the lowest loss, although its 3.64-point train-validation gap was larger than the gaps at the lower learning rates. Training-time differences among these identically sized models were treated as runtime variation rather than an effect of learning rate.

### Network-size comparison

At the fixed learning rate of `0.001`, the large network achieved the highest validation accuracy at 88.99%. The small, baseline, and large models achieved 88.35%, 88.77%, and 88.99%, respectively. Increasing capacity from 105,866 to 1,682,954 parameters therefore improved validation accuracy by only 0.64 percentage points. The corresponding generalization gap increased from 0.34 to 1.01 percentage points. These results showed diminishing returns from additional capacity and modest evidence of increased overfitting. Measured training times remained similar on the T4 GPU, ranging from 114.97 to 116.98 seconds.

![Validation loss and accuracy by epoch](results/plots.png)


In [ ]:
lr_results = results[results["network_size"] == str(BASELINE_SIZE)].copy()
size_results = results[results["learning_rate"] == BASELINE_LR].copy()
best_lr = lr_results.loc[lr_results["validation_accuracy"].idxmax()]
best_size = size_results.loc[size_results["validation_accuracy"].idxmax()]
smallest_lr = lr_results.loc[lr_results["learning_rate"].idxmin()]
largest_lr = lr_results.loc[lr_results["learning_rate"].idxmax()]
small_network = size_results.loc[size_results["parameters"].idxmin()]
large_network = size_results.loc[size_results["parameters"].idxmax()]

print("Learning-rate comparison")
print(
    f"The best tested learning rate was {best_lr.learning_rate:g}, with validation "
    f"accuracy {best_lr.validation_accuracy:.2%} and loss {best_lr.validation_loss:.4f}."
)
print(
    f"Increasing the rate from {smallest_lr.learning_rate:g} to {largest_lr.learning_rate:g} "
    f"changed final validation accuracy from {smallest_lr.validation_accuracy:.2%} to "
    f"{largest_lr.validation_accuracy:.2%}. A very small rate can under-converge within a fixed "
    f"epoch budget, while a large rate can converge faster but may oscillate around a minimum."
)

print("\nNetwork-size comparison")
print(
    f"The best tested size was {best_size.network_size}, with {best_size.parameters:,} trainable "
    f"parameters and validation accuracy {best_size.validation_accuracy:.2%}."
)
print(
    f"The small and large networks obtained {small_network.validation_accuracy:.2%} and "
    f"{large_network.validation_accuracy:.2%} validation accuracy, respectively. Their training "
    f"times were {small_network.train_time_s:.2f}s and {large_network.train_time_s:.2f}s. "
    f"Greater width raises representational capacity and computational cost. It improves performance "
    f"only when the additional capacity reduces underfitting more than it increases optimization "
    f"difficulty or overfitting."
)
print(
    f"The large network's train-validation accuracy gap was "
    f"{large_network.generalization_gap_pp:.2f} percentage points. A larger positive gap is "
    f"evidence of stronger overfitting."
)


## Final model selection and test result

I selected the `LR 0.01` configuration because it achieved the highest validation accuracy at 91.96%. I then evaluated it once on the held-out test set. It achieved 91.50% test accuracy and 0.2516 test loss in 1.783 seconds. Test accuracy was 0.46 percentage points below validation accuracy, indicating consistent generalization to held-out data.


In [ ]:
selected = results.loc[results["validation_accuracy"].idxmax()]
selected_size = tuple(int(value) for value in selected.network_size.strip("()").split(","))
selected_model = ConfigurableCNN(*selected_size).to(device)
selected_model.load_state_dict(states[selected.configuration])
test_loss, test_accuracy, test_time = evaluate(selected_model, test_loader)

print(f"Selected configuration: {selected.configuration}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test evaluation time: {test_time:.3f}s")


## Limitations

I used one seeded run for each configuration, so this experiment did not estimate variation across random initializations. The conclusions were limited to the tested hyperparameter ranges, fixed 10-epoch budget, SGD optimizer, and current hardware. Repeated runs with confidence intervals would be required to support stronger generalization claims.


## Conclusion

This experiment demonstrated that learning rate had a substantially greater effect on performance than network width under the fixed 10-epoch training budget. The `0.01` learning rate produced the strongest validation result, improving accuracy from 80.65% at `0.0001` to 91.96% while reducing validation loss from 0.5356 to 0.2314. In contrast, increasing model size from 105,866 to 1,682,954 parameters improved validation accuracy by only 0.64 percentage points and increased the train-validation gap from 0.34 to 1.01 percentage points. These findings indicated diminishing returns from additional capacity and showed that effective optimization was more consequential than model size within the tested conditions. The selected `LR 0.01` model achieved 91.50% accuracy on the untouched test set, only 0.46 percentage points below its validation accuracy. This close agreement supports the conclusion that the selected configuration generalized reliably, while the stated limitations prevented claims beyond the tested hyperparameter ranges and single seeded run.
